# Setup Environment

In [1]:
import torch
import os



In [2]:
# Verify GPU is avialable - NOTE:  You must have a GPU to run this code
print(torch.cuda.is_available())
print(f"Number of GPUs: {torch.cuda.device_count()}")

True
Number of GPUs: 1


In [3]:
# Setup required libraries
!pip install timm==0.4.12 ftfy==6.1.1 ninja==1.10.2 opensimplex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# Clone the Style3GAN repository and verify it exists
repo_url = "https://github.com/NVlabs/stylegan3.git"
repo_dir = "/content/stylegan3"

# Remove the directory if it already exists to avoid conflicts
if os.path.exists(repo_dir):
    !rm -rf {repo_dir}

# Clone the repo
!git clone {repo_url} {repo_dir}

# Change directory to the repo
%cd {repo_dir}

# Verify the contents
!ls

Cloning into '/content/stylegan3'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 212 (delta 99), reused 90 (delta 90), pack-reused 49 (from 1)
Receiving objects: 100% (212/212), 4.16 MiB | 20.39 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/stylegan3
avg_spectra.py	 dnnlib      environment.yml  gui_utils    metrics	training       viz
calc_metrics.py  Dockerfile  gen_images.py    legacy.py    README.md	train.py
dataset_tool.py  docs	     gen_video.py     LICENSE.txt  torch_utils	visualizer.py


In [5]:
# Prepare Image data using StyleGan dataset tool
#   NOTE:  IMAGES YOU ARE USING TO FINE TUNE THE GAN MUST BE IN THE SOURCE
CMD = "python /content/stylegan3/dataset_tool.py "\
  "--source /content/data/normal "\
  "--dest /content/data/dataset/normal"

!{CMD}

100% 500/500 [00:01<00:00, 460.93it/s]


In [ ]:
# Command for clearing out newly created dataset if things go wrong
#!rm -R /content/data/dataset/normal*

# Fine-Tune Model

In [6]:
# Modify these to suit your needs
EXPERIMENTS = "/content/data/experiments/normal"
DATA = "/content/data/dataset/normal"
SNAP = 10  # Number of ticks (1,000 images) to execute before Saving a model checkpoint 

# Build the command and run it
cmd = f"/usr/bin/python3 /content/stylegan3/train.py "\
  f"--snap {SNAP} --outdir {EXPERIMENTS} --data {DATA} --cfg stylegan2 --gpus 1 --batch 32 --gamma 6.6 "
!{cmd}


Training options:
{
  "G_kwargs": {
    "class_name": "training.networks_stylegan2.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 8
    },
    "channel_base": 32768,
    "channel_max": 512,
    "fused_modconv_default": "inference_only"
  },
  "D_kwargs": {
    "class_name": "training.networks_stylegan2.Discriminator",
    "block_kwargs": {
      "freeze_layers": 0
    },
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 32768,
    "channel_max": 512
  },
  "G_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.002
  },
  "D_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.002
  },
  "loss_kwargs": {
    "class_name": "training.loss.StyleGAN2Loss",
    "r1_gamma": 6.6,
    "style_mixing_prob": 0.9,
    "pl_weight": 2,
    "pl_no_weight_grad

In [ ]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

2.5.1+cu124
0.20.1+cu124
